# Avro Basics — 03: Schema Definition


Avro schemas are defined in JSON. Understanding Avro types is essential
for building correct schemas for your data.

Topics: primitive types, records, arrays, maps, unions, enums, fixed,
converting between Avro JSON schema and Spark StructType.


In [1]:
import os, time, pathlib, shutil, random, datetime, json as pyjson
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/avro_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('avro-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Avro JAR: spark-avro_2.13-4.0.2")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/09 13:06:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | Avro JAR: spark-avro_2.13-4.0.2


## Step 1 — Avro Primitive Types



In [2]:

print("""
Avro primitive types:
  null    → NullType
  boolean → BooleanType
  int     → IntegerType
  long    → LongType
  float   → FloatType
  double  → DoubleType
  bytes   → BinaryType
  string  → StringType

Logical types (annotations on primitives):
  {"type":"long","logicalType":"timestamp-micros"} → TimestampType
  {"type":"int", "logicalType":"date"}             → DateType
  {"type":"long","logicalType":"time-micros"}      → no direct Spark mapping
""")



Avro primitive types:
  null    → NullType
  boolean → BooleanType
  int     → IntegerType
  long    → LongType
  float   → FloatType
  double  → DoubleType
  bytes   → BinaryType
  string  → StringType

Logical types (annotations on primitives):
  {"type":"long","logicalType":"timestamp-micros"} → TimestampType
  {"type":"int", "logicalType":"date"}             → DateType
  {"type":"long","logicalType":"time-micros"}      → no direct Spark mapping



## Step 2 — Record Type (Nested Struct)



In [3]:

import json as pyjson

RECORD_SCHEMA = pyjson.dumps({
    "type": "record",
    "name": "Order",
    "namespace": "com.example",
    "fields": [
        {"name": "order_id",  "type": "long"},
        {"name": "customer",  "type": {
            "type": "record", "name": "Customer",
            "fields": [
                {"name": "id",      "type": "long"},
                {"name": "name",    "type": "string"},
                {"name": "country", "type": "string"},
            ]
        }},
        {"name": "amount",    "type": "double"},
        {"name": "status",    "type": "string"},
    ]
})

nested_data = spark.createDataFrame([
    (1, (101, "Alice", "US"), 999.99, "confirmed"),
    (2, (102, "Bob",   "UK"), 499.99, "shipped"),
], ["order_id","customer","amount","status"])

OUT = f"{DATA_DIR}/nested_record"
nested_data.write.format("avro").option("avroSchema", RECORD_SCHEMA).mode("overwrite").save(OUT)
print("Written nested Avro record:")
spark.read.format("avro").load(OUT).show(truncate=False)
spark.read.format("avro").load(OUT).printSchema()


26/04/09 13:06:07 WARN TaskSetManager: Lost task 1.0 in stage 0.0 (TID 1) (172.18.0.4 executor 1): org.apache.spark.SparkException: [TASK_WRITE_FAILED] Task failed while writing rows to file:/workspace/data/avro_basics/nested_record/_temporary/0/_temporary/attempt_202604091306048783173615308330345_0000_m_000001_1/part-00001-8bd4e3b3-ab21-4401-8090-bd84f826f305-c000.snappy.avro. SQLSTATE: 58030
	at org.apache.spark.sql.errors.QueryExecutionErrors$.taskFailedWhileWritingRowsError(QueryExecutionErrors.scala:772)
	at org.apache.spark.sql.execution.datasources.FileFormatDataWriter.enrichWriteError(FileFormatDataWriter.scala:92)
	at org.apache.spark.sql.execution.datasources.FileFormatDataWriter.writeWithMetrics(FileFormatDataWriter.scala:103)
	at org.apache.spark.sql.execution.datasources.FileFormatDataWriter.writeWithIterator(FileFormatDataWriter.scala:111)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.$anonfun$executeTask$1(FileFormatWriter.scala:406)
	at org.apache.spa

Py4JJavaError: An error occurred while calling o77.save.
: org.apache.spark.SparkException: [TASK_WRITE_FAILED] Task failed while writing rows to file:/workspace/data/avro_basics/nested_record/_temporary/0/_temporary/attempt_202604091306048783173615308330345_0000_m_000001_7/part-00001-8bd4e3b3-ab21-4401-8090-bd84f826f305-c000.snappy.avro. SQLSTATE: 58030
	at org.apache.spark.sql.errors.QueryExecutionErrors$.taskFailedWhileWritingRowsError(QueryExecutionErrors.scala:772)
	at org.apache.spark.sql.execution.datasources.FileFormatDataWriter.enrichWriteError(FileFormatDataWriter.scala:92)
	at org.apache.spark.sql.execution.datasources.FileFormatDataWriter.writeWithMetrics(FileFormatDataWriter.scala:103)
	at org.apache.spark.sql.execution.datasources.FileFormatDataWriter.writeWithIterator(FileFormatDataWriter.scala:111)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.$anonfun$executeTask$1(FileFormatWriter.scala:406)
	at org.apache.spark.util.Utils$.tryWithSafeFinallyAndFail

## Step 3 — Array and Map Types



In [ ]:

arr_schema = pyjson.dumps({
    "type": "record", "name": "Product",
    "fields": [
        {"name": "id",     "type": "long"},
        {"name": "tags",   "type": {"type": "array",  "items": "string"}},
        {"name": "prices", "type": {"type": "map",    "values": "double"}},
    ]
})

arr_data = spark.createDataFrame([
    (1, ["tech","premium"], {"USD": 999.99, "EUR": 919.99}),
    (2, ["budget"],         {"USD": 29.99,  "EUR": 27.99}),
], ["id","tags","prices"])

OUT_ARR = f"{DATA_DIR}/arrays_maps"
arr_data.write.format("avro").option("avroSchema", arr_schema).mode("overwrite").save(OUT_ARR)
print("Avro with arrays and maps:")
spark.read.format("avro").load(OUT_ARR).show(truncate=False)


## Step 4 — Union (Nullable Fields)

Union with null is how Avro represents nullable fields.

In [ ]:

union_schema = pyjson.dumps({
    "type": "record", "name": "Event",
    "fields": [
        {"name": "id",      "type": "long"},
        {"name": "value",   "type": ["null","double"],  "default": None},  # nullable double
        {"name": "label",   "type": ["null","string"],  "default": None},  # nullable string
        {"name": "tag",     "type": ["string","null"],  "default": "unknown"},  # string with null fallback
    ]
})

union_data = spark.createDataFrame([
    (1, 99.99, "gold",   "vip"),
    (2, None,  None,     "standard"),
    (3, 49.99, "silver", None),
], ["id","value","label","tag"])

OUT_U = f"{DATA_DIR}/unions"
union_data.write.format("avro").option("avroSchema", union_schema).mode("overwrite").save(OUT_U)
df_u = spark.read.format("avro").load(OUT_U)
print("Avro union (nullable fields):")
df_u.show()
df_u.printSchema()
print("["null","string"] in Avro → StringType nullable=true in Spark")


## Step 5 — StructType to Avro Schema Conversion



In [ ]:

from pyspark.sql.avro.functions import to_avro, from_avro

# Spark can serialize a column as Avro binary using to_avro
sample = spark.createDataFrame([(1,"Alice",99.99),(2,"Bob",49.99)], ["id","name","amount"])
avro_binary = sample.select(to_avro(F.struct("*")).alias("avro_bytes"))
print("Serialized as Avro binary column:")
avro_binary.show(truncate=False)
print("This is the Kafka Avro message format (without schema registry header).")


## Summary



In [ ]:

print("""
Avro type → Spark type mapping:
  null          → NullType
  boolean       → BooleanType
  int           → IntegerType
  long          → LongType
  float         → FloatType
  double        → DoubleType
  string        → StringType
  bytes         → BinaryType
  array[T]      → ArrayType(T)
  map[string,T] → MapType(StringType, T)
  record        → StructType
  ["null","T"]  → T (nullable)  ← most common nullable pattern
  enum          → StringType
  fixed         → BinaryType

  logicalType:date            → DateType
  logicalType:timestamp-micros → TimestampType
""")
